### Load packages

Note: before running this notebook, you must have the dataretrieval library installed:

    pip install dataretrieval

And you must have an API key for the USGS-NWIS site, which you can sign up for here:

    https://api.waterdata.usgs.gov/signup/

Once you have your API key, save it using powershell or the command prompt (for Windows, at least):

    setx API_USGS_PAT "your_token_here"

The data dictionary for the USGS NWIS API is here:

    https://api.waterdata.usgs.gov/ogcapi/v0/collections/daily/schema

And the specific parameter codes are here:

    https://api.waterdata.usgs.gov/ogcapi/v0/collections/parameter-codes/items

In [1]:
import os
import requests
import pandas as pd
from io import StringIO


### Set parameters

In [2]:
# Connect to API

usgs_token = os.getenv("API_USGS_PAT")
if usgs_token is None:
    raise RuntimeError("API_USGS_PAT environment variable not set")

In [13]:
# USGS NWIS parameters

# List of parameter codes with comments for readability
usgs_params = [
    "00003",  # sensor depth below surface
    "00010",  # water temperature
    "00060",  # discharge / water flow
    "00065",  # water level / gage height
    "00400",  # pH
    "00095",  # specific conductance (proxy for salinity)
    # Tests
    # "00059",  # specific conductance (proxy for salinity)
    # "00061",  # specific conductance (proxy for salinity)
    # "00403",  # specific conductance (proxy for salinity)
    # "00408",  # specific conductance (proxy for salinity)
]

# Dictionary of parameter codes and names
usgs_param_names = {
    "00003": "Sensor Depth (ft)",
    "00010": "Water Temp (°C)",
    "00060": "Discharge (cfs)",
    "00065": "Water Level (ft)",
    "00400": "pH",
    "00095": "Specific Conductance (muS/cm)",
    "00059": "Test2",
    "00061": "Test3",
    "00403": "Test4",
    "00408": "Test5",
}

# Convert to string for the API request
param_codes = ",".join(usgs_params)
print(param_codes)

site = "04294500"  # Lake Champlain 
start_date = "2012-01-01"
end_date = "2022-12-31"

# Instantaneous values
# url = "https://nwis.waterservices.usgs.gov/nwis/iv/"
# Daily values
url = "https://nwis.waterservices.usgs.gov/nwis/dv/"

00003,00010,00060,00065,00400,00095


### Get data from API

In [4]:
# Request parameters

params = {
    "format": "rdb", # rdb = CSV
    "sites": site,
    "parameterCd": param_codes,
    "startDT": start_date,
    "endDT": end_date,
    "siteStatus": "all"
}

headers = {
    "Authorization": f"Bearer {usgs_token}",
    "Accept": "application/json"
}

In [5]:
# Send request

response = requests.get(url, params=params, headers=headers, timeout=30)
response.raise_for_status()


In [6]:
# Read CSV, ignore comment lines
response_raw_df = pd.read_csv(StringIO(response.text), sep="\t", comment="#")

In [7]:
response_raw_df.columns

Index(['agency_cd', 'site_no', 'datetime', '219530_00010_00001',
       '219530_00010_00001_cd', '219535_00010_00002', '219535_00010_00002_cd',
       '325162_00010_00003', '325162_00010_00003_cd', '65129_00095_00001',
       '65129_00095_00001_cd', '65130_00095_00002', '65130_00095_00002_cd',
       '65131_00095_00003', '65131_00095_00003_cd'],
      dtype='object')

In [8]:
response_raw_df.head()

,agency_cd,site_no,datetime,219530_00010_00001,219530_00010_00001_cd,219535_00010_00002,219535_00010_00002_cd,325162_00010_00003,325162_00010_00003_cd,65129_00095_00001,65129_00095_00001_cd,65130_00095_00002,65130_00095_00002_cd,65131_00095_00003,65131_00095_00003_cd
0,5s,15s,20d,14n,10s,14n,10s,14n,10s,14n,10s,14n,10s,14n,10s
1,USGS,04294500,2014-09-30,18.7,A,17.2,A,17.9,A,NaN,NaN,NaN,NaN,NaN,NaN
2,USGS,04294500,2014-10-01,17.2,A,15.8,A,16.5,A,179,A,170,A,173,A
3,USGS,04294500,2014-10-02,17.0,A,15.8,A,16.4,A,177,A,170,A,174,A
4,USGS,04294500,2014-10-03,17.3,A,16.5,A,16.9,A,179,A,173,A,175,A


In [9]:
response_raw_df.to_csv(r"..\data\raw_files\usgs_response_test.csv")

In [14]:
# pivot the response data

response_df = response_raw_df.copy()

# Strip whitespace from column names
response_df.columns = response_df.columns.str.strip()

# Convert datetime column safely
# response_df["datetime"] = pd.to_datetime(response_df["datetime"], errors="coerce")
response_df["datetime"] = pd.to_datetime(
    response_df["datetime"], 
    format="%Y-%m-%d",  # matches NWIS date format
    errors="coerce"
)

# Drop rows where datetime could not be parsed (placeholder/code rows)
response_df = response_df.dropna(subset=["datetime"])

# Keep only numeric value columns (ignore _cd/qualifier columns and agency_cd)
value_cols = [c for c in response_df.columns if not c.endswith("_cd") and c not in ["agency_cd", "datetime"]]

# Convert numeric columns safely
for col in value_cols:
    response_df[col] = pd.to_numeric(response_df[col], errors="coerce")

# Keep only datetime + value columns
response_df = response_df[["datetime"] + value_cols].copy()

# Set datetime as index
response_df.set_index("datetime", inplace=True)

# Rename columns to readable parameter names
col_map = {}
for col in value_cols:
    parts = col.split("_")  # sitecode_parametercode_sensor
    param_cd = parts[1]
    if param_cd in usgs_param_names:
        col_map[col] = usgs_param_names[param_cd]
response_df.rename(columns=col_map, inplace=True)

# After renaming columns to readable names
response_df.rename(columns=col_map, inplace=True)

# Collapse any duplicate columns by taking the mean
# response_df = response_df.groupby(response_df.columns, axis=1).mean()

response_df.head()


,site_no,Water Temp (°C),Water Temp (°C),Water Temp (°C),Specific Conductance (muS/cm),Specific Conductance (muS/cm),Specific Conductance (muS/cm)
datetime,,,,,,,
2014-09-30,4294500,18.7,17.2,17.9,NaN,NaN,NaN
2014-10-01,4294500,17.2,15.8,16.5,179.0,170.0,173.0
2014-10-02,4294500,17.0,15.8,16.4,177.0,170.0,174.0
2014-10-03,4294500,17.3,16.5,16.9,179.0,173.0,175.0
2014-10-04,4294500,17.1,16.4,16.8,191.0,161.0,176.0
